In [4]:
# read parquet as df
import pandas as pd
df = pd.read_parquet("text_test.parquet")



In [5]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import normalize

# ---- CONFIG ----
fraud_prefix = "fraud_txt_emb_"
real_prefix  = "real_txt_emb_"
label_col    = "label"

# ---- 1. Extract embedding columns ----
fraud_cols = sorted([c for c in df.columns if c.startswith(fraud_prefix)])
real_cols  = sorted([c for c in df.columns if c.startswith(real_prefix)])

assert len(fraud_cols) > 0, "No fraud embedding columns found."
assert len(real_cols)  > 0, "No real embedding columns found."
assert len(fraud_cols) == len(real_cols), "Embedding dimensions mismatch."

# ---- 2. Convert to numpy arrays ----
fraud_emb = df[fraud_cols].values.astype(np.float32)
real_emb  = df[real_cols].values.astype(np.float32)

# ---- 3. L2 normalize (important for cosine similarity) ----
fraud_emb = normalize(fraud_emb, axis=1)
real_emb  = normalize(real_emb, axis=1)

# ---- 4. Cosine similarity ----
cos_sim = np.sum(fraud_emb * real_emb, axis=1)

# ---- 5. ROC AUC ----
y_true = df[label_col].values
roc_auc = roc_auc_score(y_true, cos_sim)

print(f"ROC AUC (cosine similarity): {roc_auc:.6f}")

ROC AUC (cosine similarity): 0.683667


In [13]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score

# ----------------------------
# CONFIG
# ----------------------------
pt_path = "best_model_by_val_trial_1_single_run.pt"  # or single_run_model.pt
label_col = "label"

fraud_prefix = "fraud_txt_emb_"
real_prefix  = "real_txt_emb_"

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

# ----------------------------
# LOAD CHECKPOINT / STATE_DICT (robust)
# ----------------------------
obj = torch.load(pt_path, map_location="cpu")

if isinstance(obj, dict) and "model_state" in obj:
    state_dict = obj["model_state"]
else:
    state_dict = obj  # raw state_dict

# Must contain these keys
for k in ["head.0.weight", "head.0.bias", "head.2.weight", "head.2.bias"]:
    if k not in state_dict:
        raise KeyError(f"Expected key '{k}' not found. Keys present (sample): {list(state_dict.keys())[:20]}")

w0 = state_dict["head.0.weight"]
w2 = state_dict["head.2.weight"]

hidden_dim = int(w0.shape[0])
in_dim     = int(w0.shape[1])
out_dim    = int(w2.shape[0])

print(f"[INFO] Inferred dims from checkpoint: in_dim={in_dim}, hidden_dim={hidden_dim}, out_dim={out_dim}")

# ----------------------------
# DEFINE MODEL THAT MATCHES STATE_DICT
# ----------------------------
class SiameseEmbeddingModel(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, out_dim):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x1, x2):
        z1 = self.head(x1)
        z2 = self.head(x2)
        return z1, z2

model = SiameseEmbeddingModel(in_dim, hidden_dim, out_dim).to(device)
model.load_state_dict(state_dict, strict=True)
model.eval()
print("[INFO] Model loaded.")

# ----------------------------
# HELPERS
# ----------------------------
def _to_float32_matrix(df_sub: pd.DataFrame) -> np.ndarray:
    return df_sub.to_numpy(dtype=np.float32, copy=False)

@torch.no_grad()
def roc_auc_from_two_mats(mat1: np.ndarray, mat2: np.ndarray, y_true: np.ndarray, batch_size: int = 8192) -> float:
    assert mat1.shape == mat2.shape
    n = mat1.shape[0]

    sims_all = []
    for start in range(0, n, batch_size):
        end = min(n, start + batch_size)
        x1 = torch.from_numpy(mat1[start:end]).to(device)
        x2 = torch.from_numpy(mat2[start:end]).to(device)

        z1, z2 = model(x1, x2)
        z1 = F.normalize(z1, dim=1)
        z2 = F.normalize(z2, dim=1)

        sims = F.cosine_similarity(z1, z2, dim=1)
        sims_all.append(sims.detach().cpu())

    sims = torch.cat(sims_all, dim=0).numpy()
    return float(roc_auc_score(y_true, sims))

@torch.no_grad()
def roc_auc_raw_cosine(mat1: np.ndarray, mat2: np.ndarray) -> np.ndarray:
    # raw cosine on inputs (for sanity)
    x1 = F.normalize(torch.from_numpy(mat1), dim=1)
    x2 = F.normalize(torch.from_numpy(mat2), dim=1)
    return F.cosine_similarity(x1, x2, dim=1).numpy()

# ----------------------------
# LABELS
# ----------------------------
y = df[label_col].to_numpy()
y = y.astype(np.int32, copy=False)

# ----------------------------
# (A) PREFIX-BASED: fraud_txt_emb_*, real_txt_emb_*
# ----------------------------
fraud_cols = sorted([c for c in df.columns if c.startswith(fraud_prefix)])
real_cols  = sorted([c for c in df.columns if c.startswith(real_prefix)])

print(f"[INFO] Prefix cols: fraud={len(fraud_cols)}, real={len(real_cols)}")

if len(fraud_cols) == in_dim and len(real_cols) == in_dim:
    fraud_mat = _to_float32_matrix(df[fraud_cols])
    real_mat  = _to_float32_matrix(df[real_cols])

    auc_proj_prefix = roc_auc_from_two_mats(fraud_mat, real_mat, y)
    auc_raw_prefix  = roc_auc_score(y, roc_auc_raw_cosine(fraud_mat, real_mat))

    print(f"ROC AUC (RAW cosine, prefix inputs):   {auc_raw_prefix:.6f}")
    print(f"ROC AUC (MODEL cosine, prefix inputs): {auc_proj_prefix:.6f}")
else:
    print(f"[WARN] Prefix-based eval skipped because prefix dims != in_dim ({in_dim}).")

# ----------------------------
# (B) EVALUATOR-STYLE SLICES (generalized to in_dim)
#     This matches your Evaluator approach if your parquet is laid out like:
#     [names..., label, then x1, then x2]
# ----------------------------
start_x1 = 3
start_x2 = start_x1 + in_dim
end_x2   = start_x2 + in_dim

if df.shape[1] >= end_x2:
    x1_slice = df.iloc[:, start_x1:start_x2]
    x2_slice = df.iloc[:, start_x2:end_x2]

    x1_mat = _to_float32_matrix(x1_slice)
    x2_mat = _to_float32_matrix(x2_slice)

    auc_proj_slice = roc_auc_from_two_mats(x1_mat, x2_mat, y)
    auc_raw_slice  = roc_auc_score(y, roc_auc_raw_cosine(x1_mat, x2_mat))

    print(f"ROC AUC (RAW cosine, slice inputs):    {auc_raw_slice:.6f}")
    print(f"ROC AUC (MODEL cosine, slice inputs):  {auc_proj_slice:.6f}")
else:
    print(f"[WARN] Slice-based eval skipped: df has {df.shape[1]} cols, need at least {end_x2}.")

[INFO] Inferred dims from checkpoint: in_dim=768, hidden_dim=1024, out_dim=768
[INFO] Model loaded.
[INFO] Prefix cols: fraud=768, real=768
ROC AUC (RAW cosine, prefix inputs):   0.683667
ROC AUC (MODEL cosine, prefix inputs): 0.668188
ROC AUC (RAW cosine, slice inputs):    0.683667
ROC AUC (MODEL cosine, slice inputs):  0.973995
